In [ ]:
import pandas as pd
import numpy as np
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

: 

In [ ]:
import os
import glob
import pandas as pd

def calc_vibration_mean(file_path):
    df = pd.read_csv(file_path, encoding='cp949')

    df['측정시간'] = pd.to_datetime(
        df['측정시간'],
        format='%Y-%m-%d_%H:%M:%S',
        errors='coerce'
    )

    iso = df['측정시간'].dt.isocalendar()
    df['week'] = iso['year'].astype(str) + '-' + iso['week'].astype(str)

    df = df.loc[~df['자치구'].isin(['Seoul_Grand_Park', 'mobile_type'])].copy()

    value_cols = [
        '소음 평균(dB)',
        '진동(x) 평균(mm/s)',
        '진동(y) 평균(mm/s)',
        '진동(z) 평균(mm/s)'
    ]

    for col in value_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    result = (
        df.groupby(['week', '자치구', '지역'])[value_cols]
        .mean()
        .reset_index()
    )

    result['file_name'] = os.path.basename(file_path)

    return result

folder_path = r'C:\8ssible-Healing-Seoul-Analysis\branch_SG\8SSIBLE_SG_DATA\S-DoT_NATURE_2023년(2023.01.01~2024.01.01)'

file_list = sorted(glob.glob(os.path.join(folder_path, 'S-DoT_NATURE_*.csv')))

all_result = pd.concat(
    [calc_vibration_mean(file) for file in file_list],
    ignore_index=True
)

print(all_result)

In [ ]:
all_result

In [ ]:
all_result = all_result.loc[~all_result['week'].isin(['2022-52', '2024-1'])].copy()

print(all_result['week'].unique())

In [ ]:
col = all_result['진동(x) 평균(mm/s)']

print('전체 개수:', len(col))
print('값 있는 개수:', col.count())
print('결측치 개수:', col.isna().sum())

In [ ]:
missing_by_region = (
    all_result.groupby('지역')['진동(x) 평균(mm/s)']
    .agg(
        total_count='size',
        missing_count=lambda x: x.isna().sum()
    )
    .reset_index()
)

missing_by_region['missing_ratio'] = (
    missing_by_region['missing_count'] / missing_by_region['total_count']
)

missing_by_region = missing_by_region.sort_values(
    by='missing_count',
    ascending=False
)

print(missing_by_region.sort_values('total_count', ascending=False).head(10))


In [ ]:
import os
import glob
import pandas as pd

def calc_monthly_mean(file_path):
    df = pd.read_csv(file_path, encoding='cp949')

    df['측정시간'] = pd.to_datetime(
        df['측정시간'],
        format='%Y-%m-%d_%H:%M:%S',
        errors='coerce'
    )

    df = df.dropna(subset=['측정시간']).copy()
    df = df.loc[~df['자치구'].isin(['Seoul_Grand_Park', 'mobile_type'])].copy()

    df['month'] = (
        df['측정시간'].dt.year.astype(str)
        + '.'
        + df['측정시간'].dt.month.astype(str)
    )

    value_cols = [
        '소음 평균(dB)',
        '진동(x) 평균(mm/s)',
        '진동(y) 평균(mm/s)',
        '진동(z) 평균(mm/s)'
    ]

    for col in value_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    result = (
        df.groupby(['month', '자치구', '지역'])[value_cols]
        .mean()
        .reset_index()
    )

    return result

folder_path = r'C:\8ssible-Healing-Seoul-Analysis\branch_SG\8SSIBLE_SG_DATA\S-DoT_NATURE_2023년(2023.01.01~2024.01.01)'
file_list = sorted(glob.glob(os.path.join(folder_path, 'S-DoT_NATURE_*.csv')))

all_result = pd.concat(
    [calc_monthly_mean(file) for file in file_list],
    ignore_index=True
)

all_result = all_result.loc[all_result['month'] != '2024.1'].copy()


In [ ]:
all_result